In [40]:
import tensorflow
import keras
from keras.layers import Dense, Dropout
from keras.models import Sequential
import pandas as pd
from sklearn.model_selection import train_test_split

In [41]:
df = pd.read_csv('../Dataset/diabetes.csv')
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [42]:
x = df.iloc[:,0:-1]
y = df.iloc[:,-1]

In [43]:
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size = 0.2 , random_state = 1)

In [44]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [45]:
x_train.shape

(614, 8)

## Building a simple model

In [46]:
model1 = Sequential()
model1.add(Dense(32, activation='relu', input_dim = 8)),
model1.add(Dense(1, activation = 'sigmoid'))

model1.compile(loss = 'binary_crossentropy', optimizer = 'rmsprop', metrics = ['accuracy'])
history = model1.fit(x_train, y_train, epochs = 100, validation_data = (x_test, y_test), batch_size = 32)

Epoch 1/100


/Users/rachin/Desktop/Ai/ai2.0/lib/python3.13/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.6547 - loss: 0.7015 - val_accuracy: 0.6494 - val_loss: 0.6763
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6629 - loss: 0.6398 - val_accuracy: 0.6494 - val_loss: 0.6263
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6889 - loss: 0.5997 - val_accuracy: 0.6883 - val_loss: 0.5885
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6987 - loss: 0.5691 - val_accuracy: 0.7013 - val_loss: 0.5617
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7150 - loss: 0.5459 - val_accuracy: 0.7143 - val_loss: 0.5405
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7215 - loss: 0.5287 - val_accuracy: 0.7338 - val_loss: 0.5248
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7313 - loss: 0.5152 - val_accuracy: 0.7532 - val_loss: 0.5114
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7329 - loss: 0.5034 - val_accuracy: 0.7792 - val_loss: 0.5

In [47]:
# How to select appropriate Optimizer
# How to select no of layer
# How to select no. of node in a layer
#All in a model

In [48]:
import keras_tuner as kt

In [53]:
def build_model(hp):
    model = Sequential()
    model.add(Dense(32, activation = 'relu', input_dim = 8))
    model.add(Dense(1, activation = 'sigmoid'))

    opt = hp.Choice('optimizer', values = ['adam', 'rmsprop', 'adadelta', 'sgd'])
    model.compile(loss = 'binary_crossentropy', optimizer = opt, metrics = ['accuracy'])

    return model

In [54]:
tuner =  kt.RandomSearch(
    build_model,
    objective = 'val_accuracy',
    max_trials = 5
)

In [55]:
tuner.search(x_train, y_train, validation_data=(x_test, y_test), epochs=5)

Trial 4 Complete [00h 00m 01s]
val_accuracy: 0.7662337422370911

Best val_accuracy So Far: 0.7662337422370911
Total elapsed time: 00h 00m 04s


In [56]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'rmsprop'}

In [57]:
model = tuner.get_best_models(num_models = 1)[0]

/Users/rachin/Desktop/Ai/ai2.0/lib/python3.13/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/Users/rachin/Desktop/Ai/ai2.0/lib/python3.13/site-packages/keras/src/saving/saving_lib.py:801: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [58]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [59]:
model.fit(x_train, y_train, epochs=100, initial_epoch= 6, batch_size=32, validation_data=(x_test, y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7459 - loss: 0.5116 - val_accuracy: 0.7727 - val_loss: 0.5141
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7541 - loss: 0.4976 - val_accuracy: 0.7727 - val_loss: 0.5076
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7590 - loss: 0.4900 - val_accuracy: 0.7727 - val_loss: 0.5009
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7590 - loss: 0.4835 - val_accuracy: 0.7727 - val_loss: 0.4965
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7655 - loss: 0.4781 - val_accuracy: 0.7727 - val_loss: 0.4928
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7736 - loss: 0.4742 - val_accuracy: 0.7662 - val_loss: 0.4910
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7752 - loss: 0.4707 - val_accuracy: 0.7727 - val_loss: 0.4893
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7720 - loss: 0.4676 - val_accuracy: 0.766

## No of neurons

In [60]:
def build_model(hp):
    model = Sequential()
    units = hp.Int('units', min_value = 8, max_value = 64)
    model.add(Dense(units, activation = 'relu', input_dim = 8)),
    model.add(Dense(1, activation = 'sigmoid'))

    model.compile(loss = 'binary_crossentropy', optimizer= 'adam', metrics = ['accuracy'])

    return model

In [61]:
tuner = kt.RandomSearch( build_model,
                        objective= 'val_accuracy',
                        max_trials = 5)

/Users/rachin/Desktop/Ai/ai2.0/lib/python3.13/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [62]:
tuner.search(x_train, y_train, epochs=5, batch_size=32, validation_data=(x_test, y_test))

Trial 5 Complete [00h 00m 01s]
val_accuracy: 0.7727272510528564

Best val_accuracy So Far: 0.7857142686843872
Total elapsed time: 00h 00m 06s


In [63]:
tuner.get_best_hyperparameters()[0].values

{'units': 45}

In [64]:
model3 = tuner.get_best_models(num_models = 1)[0]

/Users/rachin/Desktop/Ai/ai2.0/lib/python3.13/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/Users/rachin/Desktop/Ai/ai2.0/lib/python3.13/site-packages/keras/src/saving/saving_lib.py:801: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [65]:
model3.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 45)             │           405 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            46 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 451 (1.76 KB)

 Trainable params: 451 (1.76 KB)

 Non-trainable params: 0 (0.00 B)

In [66]:
model.fit(x_train, y_train, epochs=100, initial_epoch= 6, batch_size=32, validation_data=(x_test, y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7948 - loss: 0.4170 - val_accuracy: 0.7987 - val_loss: 0.4820
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7964 - loss: 0.4168 - val_accuracy: 0.7987 - val_loss: 0.4820
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7932 - loss: 0.4167 - val_accuracy: 0.7987 - val_loss: 0.4825
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7980 - loss: 0.4161 - val_accuracy: 0.7987 - val_loss: 0.4829
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7980 - loss: 0.4154 - val_accuracy: 0.7922 - val_loss: 0.4830
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7980 - loss: 0.4150 - val_accuracy: 0.7987 - val_loss: 0.4834
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7948 - loss: 0.4149 - val_accuracy: 0.7987 - val_loss: 0.4834
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7948 - loss: 0.4149 - val_accuracy: 0.798

## No of hidden layer

In [70]:
def build_model(hp):
    model = Sequential()
    model.add(Dense(32, activation = 'relu', input_dim = 8))

    for i in range(hp.Int('layers', min_value = 1, max_value = 10)):
        model.add(Dense(20, activation = 'relu'))
    model.add(Dense(1, activation = 'sigmoid'))
    
    model.compile(loss = 'binary_crossentropy', optimizer = 'adam', metrics = ['accuracy'])

    return model

In [71]:
tuner = kt.RandomSearch( build_model,
                        objective= 'val_accuracy',
                        max_trials = 5)

/Users/rachin/Desktop/Ai/ai2.0/lib/python3.13/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [72]:
tuner.search(x_train, y_train, epochs=5, batch_size=32, validation_data=(x_test, y_test))

Trial 5 Complete [00h 00m 01s]
val_accuracy: 0.798701286315918

Best val_accuracy So Far: 0.8051947951316833
Total elapsed time: 00h 00m 09s


In [73]:
tuner.get_best_hyperparameters()[0].values

{'layers': 2}

In [74]:
model4 = tuner.get_best_models(num_models=1)[0]

/Users/rachin/Desktop/Ai/ai2.0/lib/python3.13/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/Users/rachin/Desktop/Ai/ai2.0/lib/python3.13/site-packages/keras/src/saving/saving_lib.py:801: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [75]:
model4.fit(x_train, y_train, epochs=100, initial_epoch= 6, batch_size=32, validation_data=(x_test, y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.7638 - loss: 0.4781 - val_accuracy: 0.8117 - val_loss: 0.4774
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7736 - loss: 0.4646 - val_accuracy: 0.8247 - val_loss: 0.4717
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7801 - loss: 0.4573 - val_accuracy: 0.8182 - val_loss: 0.4703
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7801 - loss: 0.4505 - val_accuracy: 0.8117 - val_loss: 0.4705
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7801 - loss: 0.4455 - val_accuracy: 0.8182 - val_loss: 0.4679
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7834 - loss: 0.4417 - val_accuracy: 0.8247 - val_loss: 0.4663
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7866 - loss: 0.4370 - val_accuracy: 0.8117 - val_loss: 0.4673
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7883 - loss: 0.4340 - val_accuracy: 0.811

# Full tuning for entire model

In [83]:
def build_model(hp):
    model = Sequential()
    flag = 0
    for i in range(hp.Int('num_layers', min_value = 1, max_value = 12)):
        if flag == 0:
            units = hp.Int('units'+str(i), min_value = 8, max_value = 64)
            activation = hp.Choice('activation', values = ['relu', 'sigmoid', 'tanh'])
            model.add(Dense(units, activation = activation, input_dim = 8))
            model.add(Dropout(hp.Choice('dropout'+str(i), values = [0.1, 0.2, 0.3, 0.4, 0.5] )))
            flag = 1
        else:
            units = hp.Int('units'+str(i), min_value = 8, max_value = 64)
            activation = hp.Choice('activation', values = ['relu', 'sigmoid', 'tanh'])
            model.add(Dense(units, activation = activation))
            model.add(Dropout(hp.Choice('dropout'+str(i), values = [0.1, 0.2, 0.3, 0.4, 0.5] )))


    model.add(Dense(1, activation ='sigmoid'))

    optimizer = hp.Choice('optimizer', values = ['adam', 'rmsprop', 'adadelta'])
    model.compile(loss ='binary_crossentropy', optimizer =optimizer, metrics = ['accuracy'])

    return model

    


In [84]:
tuner = kt.RandomSearch(build_model,
                        objective = 'val_accuracy',
                        max_trials = 5)

/Users/rachin/Desktop/Ai/ai2.0/lib/python3.13/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [85]:
tuner.search(x_train, y_train, epochs=5, batch_size=32, validation_data=(x_test, y_test))

Trial 5 Complete [00h 00m 03s]
val_accuracy: 0.5259740352630615

Best val_accuracy So Far: 0.6428571343421936
Total elapsed time: 00h 00m 12s


In [86]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 7,
 'units0': 33,
 'activation': 'sigmoid',
 'dropout0': 0.4,
 'optimizer': 'adam',
 'units1': 8,
 'dropout1': 0.1,
 'units2': 8,
 'dropout2': 0.1,
 'units3': 8,
 'dropout3': 0.1,
 'units4': 8,
 'dropout4': 0.1,
 'units5': 8,
 'dropout5': 0.1,
 'units6': 8,
 'dropout6': 0.1}

In [87]:
model5 =  tuner.get_best_models(num_models=1)[0]

/Users/rachin/Desktop/Ai/ai2.0/lib/python3.13/site-packages/keras/src/saving/saving_lib.py:801: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 34 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [88]:
model5.fit(x_train, y_train, epochs = 100, initial_epoch = 6, batch_size=32, validation_data = (x_test, y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.5619 - loss: 0.6838 - val_accuracy: 0.6429 - val_loss: 0.6657
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6173 - loss: 0.6604 - val_accuracy: 0.6429 - val_loss: 0.6559
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6026 - loss: 0.6689 - val_accuracy: 0.6429 - val_loss: 0.6523
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6336 - loss: 0.6524 - val_accuracy: 0.6429 - val_loss: 0.6518
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6450 - loss: 0.6529 - val_accuracy: 0.6429 - val_loss: 0.6518
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6384 - loss: 0.6616 - val_accuracy: 0.6429 - val_loss: 0.6517
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6417 - loss: 0.6600 - val_accuracy: 0.6429 - val_loss: 0.6518
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6336 - loss: 0.6577 - val_accuracy: 0.642